In [1]:

import os
import numpy as np
import cv2
np.random.seed(1337)
import gc

from keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dropout, Dense, Conv2D,MaxPool2D,BatchNormalization,GlobalAveragePooling2D
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import MultiLabelBinarizer

img_size = 128 # set the image size to 128
print(os.listdir())
dataset = os.listdir("music_dataset_spectro_full_single/train") #use as labels
labels = dataset
print(labels)

['.git', '.vscode', 'audio_separator_dataset', 'Checkpoint_tester.py', 'large separation dataset output', 'longSongSplitter.py', 'Mix_maker.py', 'model saves', 'model_classes_full.ipynb', 'model_separation(large dataset).ipynb', 'model_separation(small dataset).ipynb', 'music_dataset_spectro_full_single', 'Music_mixes_dataset_classification', 'Music_separation_dataset_large', 'music_separation_dataset_small', 'my_model_full.keras', 'my_separator_model_prototype.keras', 'prediction images', 'README.md', 'small separation dataset output', 'spectrogramMaker.py', 'spectrogramMakerMix.py', 'spectrogramMakerStem.py', 'wavs from spectrograms']
['Accordion', 'Acoustic_Guitar', 'Banjo', 'Bass_Guitar', 'Clarinet', 'cowbell', 'Dobro', 'Drum_set', 'Electric_Guitar', 'flute', 'Harmonium', 'Horn', 'Keyboard', 'Mandolin', 'Organ', 'Piano', 'Saxophone', 'Shakers', 'Tambourine', 'Trombone', 'Trumpet', 'Ukulele', 'vibraphone', 'Violin']


In [ ]:
def get_dataset_array_train(data_dir):
    data = []
    path = os.path.join("Music_mixes_dataset_classification/train/"+"mix")
    for img in os.listdir("Music_mixes_dataset_classification/train/"+"mix"): #loop through mixes
        class_num = []
        for stem in os.listdir("Music_mixes_dataset_classification/train/"+"stems"):
            if img.split("_")[0] == stem.split("_")[0]:
                temp=stem.split("_",1)
                label = temp[1].split(".")[0] #split names so the instruments are used as labels
                class_num.append(labels.index(label))

        try:
            img_arr = cv2.imread(os.path.join(path,img),0)
            resized_arr =img_arr[:img_size,:img_size]
            data.append([resized_arr,class_num])
            gc.collect()
        except Exception as e:
                print(e)
    return np.array(data,dtype="object")

In [ ]:
def get_dataset_array_test(data_dir):
    data = []
    path = os.path.join("Music_mixes_dataset_classification/test/"+"mix")
    for img in os.listdir("Music_mixes_dataset_classification/test/"+"mix"):#loop through mixes
        class_num = []
        for stem in os.listdir("Music_mixes_dataset_classification/test/"+"stems"):
            if img.split("_")[0] == stem.split("_")[0]:
                temp=stem.split("_",1)
                label = temp[1].split(".")[0]  #split names so the instruments are used as labels
                class_num.append(labels.index(label))

        try:
            img_arr = cv2.imread(os.path.join(path,img),0)
            resized_arr = img_arr[:img_size,:img_size]
            data.append([resized_arr,class_num])
            gc.collect()
        except Exception as e:
                print(e)
    return np.array(data,dtype="object")

In [ ]:
def get_dataset_array_valid(data_dir):
    data = []
    path = os.path.join("Music_mixes_dataset_classification/valid/"+"mix")
    for img in os.listdir("Music_mixes_dataset_classification/valid/"+"mix"):#loop through mixes
        class_num = []
        for stem in os.listdir("Music_mixes_dataset_classification/valid/"+"stems"):
            if img.split("_")[0] == stem.split("_")[0]:
                temp=stem.split("_",1)
                label = temp[1].split(".")[0] #split names so the instruments are used as labels
                class_num.append(labels.index(label))

        try:
            img_arr = cv2.imread(os.path.join(path,img),0)
            resized_arr = img_arr[:img_size,:img_size]
            data.append([resized_arr,class_num])
            gc.collect()
        except Exception as e:
                print(e)
    return np.array(data,dtype="object")

In [5]:
train = get_dataset_array_train("music_dataset_spectro_full_single/train/")
test = get_dataset_array_test("music_dataset_spectro_full_single/test/")
valid = get_dataset_array_valid("music_dataset_spectro_full_single/valid/")

Accordion
Acoustic_Guitar
Banjo
Bass_Guitar
Clarinet
cowbell
Dobro
Drum_set
Electric_Guitar
flute
Harmonium
Horn
Keyboard
Mandolin
Organ
Piano
Saxophone
Shakers
Tambourine
Trombone
Trumpet
Ukulele
vibraphone
Violin
Accordion
Acoustic_Guitar
Banjo
Bass_Guitar
Clarinet
cowbell
Dobro
Drum_set
Electric_Guitar
flute
Harmonium
Horn
Keyboard
Mandolin
Organ
Piano
Saxophone
Shakers
Tambourine
Trombone
Trumpet
Ukulele
vibraphone
Violin
Accordion
Acoustic_Guitar
Banjo
Bass_Guitar
Clarinet
cowbell
Dobro
Drum_set
Electric_Guitar
flute
Harmonium
Horn
Keyboard
Mandolin
Organ
Piano
Saxophone
Shakers
Tambourine
Trombone
Trumpet
Ukulele
vibraphone
Violin


In [6]:
print(train.shape) # see the shapes
print(test.shape) # see the shapes
print(valid.shape) # see the shapes

(65721, 2)
(8222, 2)
(8232, 2)


In [7]:
x_train = []
y_train = []

x_val = []
y_val = []

x_test = []
y_test = []

for feature, label in train: #loop through the train array and split into x image and y labels
    x_train.append(feature)
    y_train.append(label)

del train

for feature, label in test:#loop through the test array and split into x image and y labels
    x_test.append(feature)
    y_test.append(label)
del test

for feature, label in valid:#loop through the valid array and split into x image and y labels
    x_val.append(feature)
    y_val.append(label)
     
del valid

In [8]:
gc.collect()
x_train = np.array(x_train)/255 #normalize the images
gc.collect()
x_test = np.array(x_test)/255 #normalize the images
gc.collect()
x_val = np.array(x_val)/255 #normalize the images
gc.collect()

0

In [9]:
x_train = x_train.reshape(-1, img_size, img_size, 1) # reshape images

mlb = MultiLabelBinarizer() # to make the labels multi hot

y_train = mlb.fit_transform(y_train) # make the labels multi hot

x_val = x_val.reshape(-1, img_size, img_size, 1) # reshape images

y_val = mlb.fit_transform(y_val) # make the labels multi hot


x_test = x_test.reshape(-1, img_size, img_size, 1) # reshape images

y_test = mlb.fit_transform(y_test)  # make the labels multi hot


In [ ]:
# Model setup
model = Sequential()

#first block
model.add(Conv2D(32, (3,3), activation = 'relu', padding="same", input_shape = (img_size, img_size, 1))) # input the image shape
model.add(BatchNormalization())  # normalize image
model.add(Conv2D(32, (3,3), activation = 'relu', padding="same")) 
model.add(MaxPool2D((2,2))) # downsample image

#second block
model.add(Conv2D(64, (3,3), activation = 'relu', padding="same"))
model.add(BatchNormalization()) # normalize image
model.add(Conv2D(64, (3,3), activation = 'relu', padding="same"))
model.add(MaxPool2D((2,2))) # downsample image

#third block
model.add(Conv2D(128, (3,3), activation = 'relu', padding="same"))
model.add(BatchNormalization()) # normalize image
model.add(Conv2D(128, (3,3), activation = 'relu', padding="same"))
model.add(GlobalAveragePooling2D())

#last block
model.add(Dense(units = 512, activation = 'relu')) #dense layer
model.add(Dropout(0.4))
model.add(Dense(units = 24, activation = 'sigmoid'))

model.compile(
              optimizer = 'adam', loss = 'binary_crossentropy',
              metrics = ['binary_accuracy','accuracy']
              )
     

In [16]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 128, 128, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 128, 128, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 64, 64, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 32, 32, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 512)            │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 24)             │        12,312 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 365,688 (1.39 MB)

 Trainable params: 365,240 (1.39 MB)

 Non-trainable params: 448 (1.75 KB)

In [17]:
print(x_train.shape) 
print(y_train.shape)

(65721, 128, 128, 1)
(65721, 24)


In [20]:
learning_rate_reduction = ReduceLROnPlateau(monitor = 'val_binary_accuracy', patience = 2, verbose = 1, factor = 0.3, min_lr = 0.000001)
stop_early = EarlyStopping("val_binary_accuracy",patience = 3, verbose = 1)

In [22]:
batch_size = 32
n_epochs = 50
model.fit(x_train, y_train, batch_size = batch_size,
                    epochs = n_epochs, validation_data = (x_val, y_val),
                    callbacks = [learning_rate_reduction,stop_early], shuffle = True)

Epoch 1/50
2054/2054 ━━━━━━━━━━━━━━━━━━━━ 910s 443ms/step - binary_accuracy: 0.9413 - loss: 0.1990 - val_binary_accuracy: 0.9390 - val_loss: 0.2067 - learning_rate: 0.0010
Epoch 2/50
2054/2054 ━━━━━━━━━━━━━━━━━━━━ 900s 438ms/step - binary_accuracy: 0.9425 - loss: 0.1943 - val_binary_accuracy: 0.9405 - val_loss: 0.2068 - learning_rate: 0.0010
Epoch 3/50
2054/2054 ━━━━━━━━━━━━━━━━━━━━ 899s 438ms/step - binary_accuracy: 0.9432 - loss: 0.1912 - val_binary_accuracy: 0.9390 - val_loss: 0.2156 - learning_rate: 0.0010
Epoch 4/50
2054/2054 ━━━━━━━━━━━━━━━━━━━━ 899s 438ms/step - binary_accuracy: 0.9438 - loss: 0.1888 - val_binary_accuracy: 0.9435 - val_loss: 0.1907 - learning_rate: 0.0010
Epoch 5/50
2054/2054 ━━━━━━━━━━━━━━━━━━━━ 908s 442ms/step - binary_accuracy: 0.9442 - loss: 0.1869 - val_binary_accuracy: 0.9389 - val_loss: 0.2065 - learning_rate: 0.0010
Epoch 6/50
2054/2054 ━━━━━━━━━━━━━━━━━━━━ 0s 427ms/step - binary_accuracy: 0.9443 - loss: 0.1858
Epoch 6: ReduceLROnPlateau reducing learnin

In [ ]:
gc.collect()
print("loss of the model is - " , model.evaluate(x_test,y_test)[0])
print("Accuracy of the model is - " , model.evaluate(x_test,y_test)[1]*100 , "%")
model.save('Classifiaction_model_full_multi.keras')

257/257 ━━━━━━━━━━━━━━━━━━━━ 25s 97ms/step - binary_accuracy: 0.9447 - loss: 0.1870
loss of the model is -  0.18704481422901154
257/257 ━━━━━━━━━━━━━━━━━━━━ 24s 94ms/step - binary_accuracy: 0.9447 - loss: 0.1870
Accuracy of the model is -  94.47163343429565 %


In [24]:
predictions=model.predict(x_test)
pred_labels= np.where(predictions>0.5)

257/257 ━━━━━━━━━━━━━━━━━━━━ 25s 95ms/step


In [29]:
img = cv2.imread("music_dataset_spectro_full_single/test/Accordion/2864_Accordion.png",0)
img=cv2.resize(img,(img_size,img_size))
img= np.reshape(img,(-1, img_size, img_size, 1))
img=img/255

print(labels)
labels=np.array(labels)
pred=model.predict(img)
prediction = np.argmax(pred,1)
print(labels[prediction])


['Accordion' 'Acoustic_Guitar' 'Banjo' 'Bass_Guitar' 'Clarinet' 'cowbell'
 'Dobro' 'Drum_set' 'Electric_Guitar' 'flute' 'Harmonium' 'Horn'
 'Keyboard' 'Mandolin' 'Organ' 'Piano' 'Saxophone' 'Shakers' 'Tambourine'
 'Trombone' 'Trumpet' 'Ukulele' 'vibraphone' 'Violin']
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
['Accordion']


In [28]:
img = cv2.imread("Music_mixes_dataset_classification/test/mix/32836_mix.png",0)
img=img[:img_size,:img_size]
img= np.reshape(img,(-1, img_size, img_size, 1))
img=img/255

print(labels)
labels=np.array(labels)
pred=model.predict(img)
print(pred)
prediction = np.argmax(pred,1)
print(labels[prediction])

['Accordion', 'Acoustic_Guitar', 'Banjo', 'Bass_Guitar', 'Clarinet', 'cowbell', 'Dobro', 'Drum_set', 'Electric_Guitar', 'flute', 'Harmonium', 'Horn', 'Keyboard', 'Mandolin', 'Organ', 'Piano', 'Saxophone', 'Shakers', 'Tambourine', 'Trombone', 'Trumpet', 'Ukulele', 'vibraphone', 'Violin']
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
[[0.10341315 0.10627937 0.09382852 0.10988723 0.10617243 0.10451294
  0.10338479 0.10440937 0.11014125 0.09601973 0.10441627 0.10234208
  0.10482054 0.10566185 0.10295834 0.10562307 0.0990603  0.1074715
  0.09853418 0.10412535 0.09882335 0.10604329 0.09883285 0.11052264]]
['Violin']


In [30]:
img = cv2.imread("Music_mixes_dataset_classification/test/mix/32844_mix.png",0)
img=img[:img_size,:img_size]
img= np.reshape(img,(-1, img_size, img_size, 1))
img=img/255

print(labels)
labels=np.array(labels)
pred=model.predict(img)
print(pred)
prediction = np.argmax(pred,1)
print(labels[prediction])

['Accordion' 'Acoustic_Guitar' 'Banjo' 'Bass_Guitar' 'Clarinet' 'cowbell'
 'Dobro' 'Drum_set' 'Electric_Guitar' 'flute' 'Harmonium' 'Horn'
 'Keyboard' 'Mandolin' 'Organ' 'Piano' 'Saxophone' 'Shakers' 'Tambourine'
 'Trombone' 'Trumpet' 'Ukulele' 'vibraphone' 'Violin']
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
[[0.10041023 0.10672802 0.10585374 0.09930916 0.10549602 0.09901952
  0.08919042 0.10532605 0.09965262 0.09615943 0.09793919 0.10809336
  0.10107901 0.10486827 0.10814676 0.10306966 0.10495259 0.09776178
  0.0989021  0.11046403 0.08755416 0.09576897 0.11137535 0.10919725]]
['vibraphone']
